In [1]:
import os

In [2]:
%pwd

'e:\\Projects\\Text-Summarization\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\Projects\\Text-Summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config

In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import evaluate
import torch
import pandas as pd
from tqdm import tqdm
from textSummarizer.logging import logger

c:\Users\admin\anaconda3\envs\summary\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i: i + batch_size]

    def calculate_metric_on_test_ds(
        self, dataset, metric, model, tokenizer,
        batch_size=8, device="cuda" if torch.cuda.is_available() else "cpu",
        column_text="dialogue", column_summary="summary"
    ):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)
        ):
            inputs = tokenizer(
                article_batch,
                max_length=512,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=4,
                max_length=128
            )

            decoded_summaries = [
                tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                for s in summaries
            ]

            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        score = metric.compute(use_stemmer=True)
        return score

    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        logger.info(f"Using device: {device}")

        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)

        dataset = load_from_disk(self.config.data_path)
        metric = evaluate.load("rouge")

        score = self.calculate_metric_on_test_ds(
            dataset["test"][:10], metric, model, tokenizer,
            batch_size=2, column_text="dialogue", column_summary="summary"
        )

        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_dict = {rn: score[rn] for rn in rouge_names}  

        df = pd.DataFrame(rouge_dict, index=["t5-small"])
        df.to_csv(self.config.metric_file_name, index=False)
        logger.info(f"Evaluation completed. Metrics saved at {self.config.metric_file_name}")
        print(df)


In [10]:
try:
    config = ConfigurationManager()
    model_eval_config = config.get_model_evaluation_config()
    model_evaluator = ModelEvaluation(config=model_eval_config)
    model_evaluator.evaluate()
except Exception as e:
    raise e

YAML file loaded successfully: config\config.yaml
[2025-11-03 19:21:48,315: INFO: common: YAML file loaded successfully: config\config.yaml]
YAML file loaded successfully: params.yaml
[2025-11-03 19:21:48,327: INFO: common: YAML file loaded successfully: params.yaml]
Created directory at: artifacts
[2025-11-03 19:21:48,333: INFO: common: Created directory at: artifacts]
Created directory at: artifacts/model_evaluation
[2025-11-03 19:21:48,338: INFO: common: Created directory at: artifacts/model_evaluation]
Using device: cpu
[2025-11-03 19:21:48,341: INFO: 1334622973: Using device: cpu]


100%|██████████| 5/5 [01:15<00:00, 15.15s/it]


[2025-11-03 19:23:10,640: INFO: rouge_scorer: Using default tokenizer.]
Evaluation completed. Metrics saved at artifacts/model_evaluation/metrics.csv
[2025-11-03 19:23:11,158: INFO: 1334622973: Evaluation completed. Metrics saved at artifacts/model_evaluation/metrics.csv]
            rouge1    rouge2    rougeL  rougeLsum
t5-small  0.230143  0.045468  0.183279    0.18177
